# 🌱 HydroGrow AI — Phase 3: Feature Engineering

---

**Notebook:** `04_Feature_Engineering.ipynb`  
**Project:** HydroGrow AI Decision Support System  
**Phase:** Phase 3 (Feature Engineering)  
**Author:** HydroGrow AI Team  
**Date:** 2026-07-15  
**Version:** 1.0  

---

## 1 · Introduction & Objective

In Phase 1 and Phase 2, we cleaned our datasets and diagnosed their suitability for Machine Learning. We concluded that training models directly on the concatenated experiment datasets was not possible due to row-wise stacking, and recommended training on aggregated sheet-level datasets instead.

### 1.1 Objective
The primary objective of this notebook is to execute the data aggregation, mapping, merging, and feature engineering pipeline to build the **final flat machine learning dataset** for lettuce growth prediction. 

The target data structure is:
$$\text{One Row} = \text{One Individual Harvested Lettuce Plant}$$

We will load and combine sheet-level data from:
- `harvest` (to extract individual plant growth targets and identifiers)
- `seedlings` (to calculate experiment-level initial baseline features)
- `sensor_water_quality` (to calculate global air temperature, humidity, and CO₂ stats)
- `portable_water_quality` (to calculate system-specific pH, EC, TDS, and water temperature stats)
- `nutrients` (to calculate cumulative nutrient additions, water, and acid consumption)
- `head_diameter` (to inspect and verify time-series growth tracking)

### 1.2 Pipeline Steps
1. **Load Data**: Discover and load all required sheet-level CSV files.
2. **Column Standardization**: Define a standardized column naming schema via a mapping dictionary.
3. **Aggregate Environmental Sensor Averages**: Calculate mean, min, max, and standard deviation of air temperature, humidity, and CO₂ per experiment.
4. **Aggregate Portable Water Quality**: Pool morning and afternoon spot-checks of pH, EC, TDS, and water temperature, and calculate mean, min, max, and standard deviation per system.
5. **Process Nutrient Logs**: Compute total cumulative nutrient solution, water, and acid consumed per system.
6. **Process Seedlings Baseline**: Calculate starting average height, weight, and root length per experiment cohort.
7. **Extract Harvest Targets & Parse Head Diameter**: Parse the string head diameter dimensions (`Width*Height`) into average diameter and area, filter out system average/summary rows, and standardize crop yield metrics.
8. **Merge Datasets**: Align replicate system labels and merge all feature groups horizontally using `experiment`, `system`, and `replicate` as keys.
9. **Handle Missing Values**: Inspect and handle missing data with explicit rationale.
10. **Save Dataset & Generate Report**: Export the final CSV and generate a Feature Engineering Report.

## 2 · Import Libraries

We import standard packages for numerical calculation, dataframes manipulation, and filesystem search.

In [1]:
import os
import glob
import re
import numpy as np
import pandas as pd

# Fallback for display if run outside Jupyter
try:
    from IPython.display import display
except ImportError:
    display = print

print("Libraries successfully imported!")

Libraries successfully imported!


## 3 · Load Sheet-Level Cleaned Datasets

We load all sheet-level CSV files from `../data/processed/per_sheet/`. We will group them by experiment for structured processing.

In [2]:
def load_sheet_data(folder_path):
    """
    Loads all CSV files in the specified folder and returns a dictionary of DataFrames.
    Keys are the filenames without '.csv'.
    """
    csv_files = glob.glob(os.path.join(folder_path, "*_clean.csv"))
    datasets = {}
    for filepath in sorted(csv_files):
        basename = os.path.basename(filepath)
        key = basename.replace(".csv", "")
        try:
            df = pd.read_csv(filepath)
            datasets[key] = df
            print(f"Loaded '{key}' successfully with shape {df.shape}")
        except Exception as e:
            print(f"Failed to load {basename}: {e}")
    return datasets

sheet_data = load_sheet_data("../data/processed/per_sheet")

Loaded 'exp1_harvest_clean' successfully with shape (73, 13)
Loaded 'exp1_head_diameter_clean' successfully with shape (204, 5)
Loaded 'exp1_nutrients_acid_consumption_(ml)_clean' successfully with shape (4, 7)
Loaded 'exp1_nutrients_date_clean' successfully with shape (5, 2)
Loaded 'exp1_nutrients_nutrient_solution_addition_(a+b)_ml_clean' successfully with shape (5, 7)
Loaded 'exp1_nutrients_water_consumption_l_clean' successfully with shape (5, 7)
Loaded 'exp1_portable_water_quality_clean' successfully with shape (32, 57)
Loaded 'exp1_seedlings_clean' successfully with shape (10, 12)
Loaded 'exp1_sensor_water_quality_clean' successfully with shape (641, 13)
Loaded 'exp2_form_responses_clean' successfully with shape (7, 26)
Loaded 'exp2_harvest_clean' successfully with shape (78, 13)
Loaded 'exp2_nutrients_date_clean' successfully with shape (2, 2)
Loaded 'exp2_nutrients_nutrient_solution_addition_(a+b)_ml_clean' successfully with shape (3, 7)
Loaded 'exp2_nutrients_water_consumption

## 4 · Column Standardization System

We create a column standardization schema to resolve differences in naming conventions across experiments (e.g. `air_temp_c` vs `air_temp`, `rh_%` vs `rh%`).

Below is our mapping dictionary, which translates raw environmental and biological names to clean, unified ML feature names.

In [3]:
# Mapping dictionary for standardizing columns
COLUMN_STANDARDIZATION_MAP = {
    # Environmental names
    "air_temp_c": "air_temperature",
    "air_temp": "air_temperature",
    "rh_%": "humidity",
    "rh%": "humidity",
    "co2_ppm": "co2",
    "co2": "co2",
    
    # Portable water quality metrics
    "ph": "ph",
    "ec": "ec",
    "tds": "tds",
    "water_temp": "water_temperature",
    
    # Biological metrics
    "plant_height_cm": "plant_height_cm",
    "root_length_cm": "root_length_cm",
    "total_weight_g": "total_weight_g",
    "root_weight_g": "root_weight_g",
    "no_of_leaves": "no_of_leaves",
    "noof_leaves": "no_of_leaves"
}

print("Column standardization map created:")
for k, v in COLUMN_STANDARDIZATION_MAP.items():
    print(f"  {k:30} -> {v}")

Column standardization map created:
  air_temp_c                     -> air_temperature
  air_temp                       -> air_temperature
  rh_%                           -> humidity
  rh%                            -> humidity
  co2_ppm                        -> co2
  co2                            -> co2
  ph                             -> ph
  ec                             -> ec
  tds                            -> tds
  water_temp                     -> water_temperature
  plant_height_cm                -> plant_height_cm
  root_length_cm                 -> root_length_cm
  total_weight_g                 -> total_weight_g
  root_weight_g                  -> root_weight_g
  no_of_leaves                   -> no_of_leaves
  noof_leaves                    -> no_of_leaves


## 5 · Process Environmental Data

We aggregate the continuous time-series environmental sensor readings (air temperature, humidity, and CO₂) for each experiment. Since the sensor records only contain one dummy replicate, we calculate experiment-wide summary statistics (Mean, Min, Max, and Standard Deviation).

In [4]:
def aggregate_sensor_data(df, exp_label):
    """
    Extracts air temperature, humidity, and CO2, standardizes their names, 
    and calculates Mean, Min, Max, and Std Dev.
    """
    # Find the appropriate column names
    temp_col = 'air_temp_c' if 'air_temp_c' in df.columns else ('air_temp' if 'air_temp' in df.columns else None)
    rh_col = 'rh_%' if 'rh_%' in df.columns else ('rh%' if 'rh%' in df.columns else None)
    co2_col = 'co2_ppm' if 'co2_ppm' in df.columns else ('co2' if 'co2' in df.columns else None)
    
    metrics = {}
    # Map each variable and compute statistics
    for var_name, col in [('air_temperature', temp_col), ('humidity', rh_col), ('co2', co2_col)]:
        if col is not None:
            data = df[col].dropna()
            if len(data) > 0:
                metrics[f"env_{var_name}_mean"] = data.mean()
                metrics[f"env_{var_name}_min"] = data.min()
                metrics[f"env_{var_name}_max"] = data.max()
                metrics[f"env_{var_name}_std"] = data.std()
            else:
                metrics[f"env_{var_name}_mean"] = np.nan
                metrics[f"env_{var_name}_min"] = np.nan
                metrics[f"env_{var_name}_max"] = np.nan
                metrics[f"env_{var_name}_std"] = np.nan
        else:
            metrics[f"env_{var_name}_mean"] = np.nan
            metrics[f"env_{var_name}_min"] = np.nan
            metrics[f"env_{var_name}_max"] = np.nan
            metrics[f"env_{var_name}_std"] = np.nan
            
    return metrics

print("--- Aggregating Experiment Sensor Environments ---")
sensor_aggregates = {}
for i in [1, 2, 3]:
    key = f"exp{i}_sensor_water_quality_clean"
    if key in sheet_data:
        sensor_aggregates[f"Experiment {i}"] = aggregate_sensor_data(sheet_data[key], f"Experiment {i}")
        print(f"Experiment {i} Sensor averages:")
        print(f"  Mean Temp: {sensor_aggregates[f'Experiment {i}']['env_air_temperature_mean']:.2f}°C")
        print(f"  Mean RH  : {sensor_aggregates[f'Experiment {i}']['env_humidity_mean']:.2f}%")
        print(f"  Mean CO2 : {sensor_aggregates[f'Experiment {i}']['env_co2_mean']:.2f} ppm")

--- Aggregating Experiment Sensor Environments ---
Experiment 1 Sensor averages:
  Mean Temp: 19.93°C
  Mean RH  : 57.40%
  Mean CO2 : 456.08 ppm
Experiment 2 Sensor averages:
  Mean Temp: 22.50°C
  Mean RH  : 57.99%
  Mean CO2 : 454.28 ppm
Experiment 3 Sensor averages:
  Mean Temp: 25.46°C
  Mean RH  : 61.47%
  Mean CO2 : 450.34 ppm


### 5.1 Process Portable Water Quality Data

Portable water quality measurements are recorded in columns per replicate system, labeled `100000_replicate_X_tY_{metric}` and `140000_replicate_X_tY_{metric}`. 
We need to pool the measurements from morning (10:00 AM) and afternoon (2:00 PM) for each replicate system `replicate_X_tY` (where $X \in \{1, 2, 3\}$ and $Y \in \{1, 2\}$) and calculate the Mean, Min, Max, and Std Dev of water pH, EC, TDS, and water temperature.

We will map these replicate-tank combinations to the corresponding harvest system ID based on our key matching rules:
- **Experiment 1**: column `replicate_X_tY` corresponds to harvest system `R{Y}-T{X}`.
- **Experiment 2**: column `replicate_X_tY` corresponds to harvest system `R{X}-T{Y}`.
- **Experiment 3**: column `replicate_X_tY` corresponds to harvest system `R{X}T{Y}`.

In [5]:
def aggregate_portable_water_data(df, exp_label):
    """
    Extracts pH, EC, TDS, and water temp for replicates 1-3 and tanks t1-t2.
    Pools time points (100000 and 140000) and calculates summary statistics.
    Maps columns to harvest-compatible system names.
    """
    replicates = [1, 2, 3]
    tanks = ['t1', 't2']
    
    sys_aggregates = {}
    for rep in replicates:
        for tank in tanks:
            # Determine target system ID
            if exp_label == 'Experiment 1':
                system_id = f"R{tank[1]}-T{rep}"
            elif exp_label == 'Experiment 2':
                system_id = f"R{rep}-T{tank[1].upper()}"
            elif exp_label == 'Experiment 3':
                system_id = f"R{rep}T{tank[1].upper()}"
                
            col_pattern = f"replicate_{rep}_{tank}"
            
            # Find columns matching the system key
            ph_cols = [c for c in df.columns if col_pattern in c and c.endswith('_ph')]
            ec_cols = [c for c in df.columns if col_pattern in c and c.endswith('_ec')]
            tds_cols = [c for c in df.columns if col_pattern in c and c.endswith('_tds')]
            wt_cols = [c for c in df.columns if col_pattern in c and c.endswith('_water_temp')]
            
            metrics = {}
            # Compute metrics for each sensor reading type
            for name, cols in [('ph', ph_cols), ('ec', ec_cols), ('tds', tds_cols), ('water_temperature', wt_cols)]:
                if cols:
                    # Flatten morning and afternoon columns into a single series
                    vals = df[cols].values.flatten()
                    vals = vals[~np.isnan(vals)]  # Drop NaNs
                    
                    if len(vals) > 0:
                        metrics[f"water_{name}_mean"] = np.mean(vals)
                        metrics[f"water_{name}_min"] = np.min(vals)
                        metrics[f"water_{name}_max"] = np.max(vals)
                        metrics[f"water_{name}_std"] = np.std(vals)
                    else:
                        metrics[f"water_{name}_mean"] = np.nan
                        metrics[f"water_{name}_min"] = np.nan
                        metrics[f"water_{name}_max"] = np.nan
                        metrics[f"water_{name}_std"] = np.nan
                else:
                    metrics[f"water_{name}_mean"] = np.nan
                    metrics[f"water_{name}_min"] = np.nan
                    metrics[f"water_{name}_max"] = np.nan
                    metrics[f"water_{name}_std"] = np.nan
                    
            sys_aggregates[(exp_label, system_id)] = metrics
            
    return sys_aggregates

print("--- Aggregating Portable Water Quality Data ---")
portable_aggregates = {}
for i in [1, 2, 3]:
    key = f"exp{i}_portable_water_quality_clean"
    if key in sheet_data:
        exp_label = f"Experiment {i}"
        exp_portable_aggs = aggregate_portable_water_data(sheet_data[key], exp_label)
        portable_aggregates.update(exp_portable_aggs)
        
print(f"Portable aggregates calculated for {len(portable_aggregates)} experiment-system combinations.")

--- Aggregating Portable Water Quality Data ---
Portable aggregates calculated for 18 experiment-system combinations.


## 6 · Process Nutrient Data

We calculate the total nutrients, acid, and water additions for each replicate system. This is done by summing up the periodic values inside each system's column.
- **Total nutrient solution added** (from `nutrient_solution_addition` sheet)
- **Total water consumption** (from `water_consumption` sheet)
- **Total acid consumption** (from `acid_consumption` sheet — *Note: Experiment 2 does not have an acid consumption log; we will record NaN and handle it in the missing value step*)

In [6]:
def aggregate_nutrients(solutions_df, water_df, acid_df, exp_label):
    """
    Calculates the sum of nutrients solution, water consumption, and acid additions.
    Maps columns to harvest-compatible system names.
    """
    replicates = [1, 2, 3]
    tanks = ['t1', 't2']
    
    sys_nutrients = {}
    for rep in replicates:
        for tank in tanks:
            if exp_label == 'Experiment 1':
                system_id = f"R{tank[1]}-T{rep}"
            elif exp_label == 'Experiment 2':
                system_id = f"R{rep}-T{tank[1].upper()}"
            elif exp_label == 'Experiment 3':
                system_id = f"R{rep}T{tank[1].upper()}"
                
            col_name = f"replicate_{rep}_{tank}"
            
            # Cumulative Solution Added
            sol_sum = solutions_df[col_name].sum() if (solutions_df is not None and col_name in solutions_df.columns) else 0.0
            # Cumulative Water Consumed
            water_sum = water_df[col_name].sum() if (water_df is not None and col_name in water_df.columns) else 0.0
            # Cumulative Acid Consumed
            acid_sum = acid_df[col_name].sum() if (acid_df is not None and col_name in acid_df.columns) else np.nan
            
            sys_nutrients[(exp_label, system_id)] = {
                "total_nutrient_solution_added_ml": float(sol_sum),
                "total_water_consumption_l": float(water_sum),
                "total_acid_consumption_ml": float(acid_sum) if acid_df is not None else np.nan
            }
            
    return sys_nutrients

print("--- Aggregating Nutrient & Water Consumption logs ---")
nutrient_aggregates = {}
for i in [1, 2, 3]:
    exp_label = f"Experiment {i}"
    sol_key = f"exp{i}_nutrients_nutrient_solution_addition_(a+b)_ml_clean"
    water_key = f"exp{i}_nutrients_water_consumption_l_clean"
    
    # Standardize acid consumption sheet name variations
    acid_key = f"exp{i}_nutrients_acid_consumption_(ml)_clean" if i == 1 else f"exp{i}_nutrients_acid_consumption_ml_clean"
    
    sol_df = sheet_data.get(sol_key)
    water_df = sheet_data.get(water_key)
    acid_df = sheet_data.get(acid_key)
    
    exp_nut_aggs = aggregate_nutrients(sol_df, water_df, acid_df, exp_label)
    nutrient_aggregates.update(exp_nut_aggs)

print(f"Nutrient aggregates calculated for {len(nutrient_aggregates)} combinations.")

--- Aggregating Nutrient & Water Consumption logs ---
Nutrient aggregates calculated for 18 combinations.


## 7 · Process Seedlings Baseline Data

Since seedling parameters are baseline measurements of a random sample at transplant and do not map directly row-by-row to final plants, we calculate the average seedling **height, weight, and root length** per experiment cohort. These serve as baseline starting features.

In [7]:
seedling_baselines = {}
print("--- Calculating Seedling Baseline Averages ---")
for i in [1, 2, 3]:
    key = f"exp{i}_seedlings_clean"
    exp_label = f"Experiment {i}"
    if key in sheet_data:
        df = sheet_data[key]
        # Calculate seedling averages
        avg_height = df["plant_height_cm"].mean() if "plant_height_cm" in df.columns else np.nan
        avg_weight = df["total_weight_g"].mean() if "total_weight_g" in df.columns else np.nan
        avg_root_len = df["root_length_cm"].mean() if "root_length_cm" in df.columns else np.nan
        
        seedling_baselines[exp_label] = {
            "initial_height_mean_cm": avg_height,
            "initial_weight_mean_g": avg_weight,
            "initial_root_length_mean_cm": avg_root_len
        }
        print(f"{exp_label} Baselines:")
        print(f"  Initial Height : {avg_height:.2f} cm")
        print(f"  Initial Weight : {avg_weight:.2f} g")
        print(f"  Initial Root   : {avg_root_len:.2f} cm")

--- Calculating Seedling Baseline Averages ---
Experiment 1 Baselines:
  Initial Height : 9.75 cm
  Initial Weight : 1.22 g
  Initial Root   : 7.10 cm
Experiment 2 Baselines:
  Initial Height : 12.71 cm
  Initial Weight : 4.86 g
  Initial Root   : 6.38 cm
Experiment 3 Baselines:
  Initial Height : 14.45 cm
  Initial Weight : 4.80 g
  Initial Root   : 8.15 cm


## 8 · Process Harvest Data and Target Parsing

We process the harvest sheets (`exp1_harvest_clean`, `exp2_harvest_clean`, `exp3_harvest_clean`). 
Our tasks here are:
1. **Filter out summary rows**: Drop any rows where `plant_no` is null. This removes replicate average summary rows and trailing blank rows, leaving exactly the individual plant records.
2. **Parse Head Diameter**: Parse the string head diameter dimensions (`Width*Height` e.g., `'24*27'`) into a numeric average head diameter and rectangular canopy area.
3. **Extract Targets**: Set `total_weight_g` as our primary prediction target, and keep standardized shoot weight, height, and leaf count.

In [8]:
def parse_head_diameter_str(val):
    """
    Parses a head diameter string (e.g. '24*27' or '24') and returns (average_diameter, canopy_area).
    """
    if pd.isna(val) or val == "" or str(val).lower().strip() in ("nan", "nat", "null", "h.d (cm)"):
        return np.nan, np.nan
    
    val_str = str(val).lower().replace("cm", "").strip()
    
    # Split on standard separators
    for separator in ['*', 'x', '×']:
        if separator in val_str:
            parts = val_str.split(separator)
            try:
                w = float(parts[0].strip())
                h = float(parts[1].strip())
                return (w + h) / 2.0, w * h
            except ValueError:
                pass
                
    # Try parsing single numeric string
    try:
        d = float(val_str)
        return d, d * d
    except ValueError:
        pass
        
    return np.nan, np.nan

def process_harvest(df, exp_label):
    """
    Standardizes harvest dataset, drops summary average rows (where plant_no is NaN),
    and parses string head diameter columns.
    """
    # Drop rows where plant_no is missing (summary and trailing rows)
    df_filtered = df.dropna(subset=['plant_no']).copy()
    df_filtered['plant_no'] = df_filtered['plant_no'].astype(int)
    df_filtered['experiment'] = exp_label
    
    # Resolve names mapping
    shoot_wt_col = 'shoot_weight_after_removing_wilted_leavesg' if 'shoot_weight_after_removing_wilted_leavesg' in df_filtered.columns else 'shoot_weight_after_removing_wilted_leaves_g'
    hd_col = 'head_diameter_cm' if 'head_diameter_cm' in df_filtered.columns else 'hd_cm'
    leaf_col = 'no_of_leaves' if 'no_of_leaves' in df_filtered.columns else 'noof_leaves'
    
    # Parse head diameter
    avg_hds = []
    canopy_areas = []
    for val in df_filtered[hd_col]:
        avg_hd, area = parse_head_diameter_str(val)
        avg_hds.append(avg_hd)
        canopy_areas.append(area)
        
    df_filtered['head_diameter_average_cm'] = avg_hds
    df_filtered['canopy_area_cm2'] = canopy_areas
    
    # Assign predictions targets and clean features
    df_filtered['target_total_weight_g'] = df_filtered['total_weight_g']
    df_filtered['harvest_plant_height_cm'] = df_filtered['plant_height_cm']
    df_filtered['harvest_shoot_weight_g'] = df_filtered[shoot_wt_col]
    df_filtered['harvest_root_weight_g'] = df_filtered['root_weight_g']
    df_filtered['harvest_root_length_cm'] = df_filtered['root_length_cm']
    df_filtered['harvest_no_of_leaves'] = df_filtered[leaf_col]
    
    cols_to_keep = [
        'experiment', 'system', 'plant_no',
        'target_total_weight_g', 'harvest_plant_height_cm', 'harvest_shoot_weight_g',
        'harvest_root_weight_g', 'harvest_root_length_cm', 'harvest_no_of_leaves',
        'head_diameter_average_cm', 'canopy_area_cm2'
    ]
    
    return df_filtered[cols_to_keep]

print("--- Processing Harvest Datasets ---")
processed_harvest_dfs = []
for i in [1, 2, 3]:
    key = f"exp{i}_harvest_clean"
    if key in sheet_data:
        df = process_harvest(sheet_data[key], f"Experiment {i}")
        processed_harvest_dfs.append(df)
        print(f"Experiment {i}: Filtered out summary/blank rows. Kept {len(df)} plant rows.")
        
combined_harvest_df = pd.concat(processed_harvest_dfs, ignore_index=True)
print(f"\nCombined Harvest dataset shape: {combined_harvest_df.shape}")

--- Processing Harvest Datasets ---
Experiment 1: Filtered out summary/blank rows. Kept 72 plant rows.
Experiment 2: Filtered out summary/blank rows. Kept 72 plant rows.
Experiment 3: Filtered out summary/blank rows. Kept 72 plant rows.

Combined Harvest dataset shape: (216, 11)


## 9 · Merge Datasets

We merge all the aggregated features (sensor environment, portable water quality, cumulative nutrients, seedlings baselines) onto our individual plant harvest records.

To do so, we first extract the `replicate` index (e.g. `R1`, `R2`, `R3`) from the `system` column so that we can merge using all three required keys:
- `experiment`
- `system`
- `replicate`

In [9]:
def extract_replicate(sys_val):
    """
    Extracts replicate code from system name.
    e.g. 'R1-T1' -> 'R1'
         'R2T2' -> 'R2'
    """
    if pd.isna(sys_val) or not isinstance(sys_val, str):
        return np.nan
    sys_clean = sys_val.strip()
    if len(sys_clean) >= 2:
        return sys_clean[:2]
    return np.nan

# 1. Add replicate column to harvest dataframe
combined_harvest_df['replicate'] = combined_harvest_df['system'].apply(extract_replicate)

# 2. Build the system-level features dataframe
sys_records = []
for (exp_label, sys_id), port_metrics in portable_aggregates.items():
    nut_metrics = nutrient_aggregates.get((exp_label, sys_id), {})
    
    # Compile all features for this experiment and system
    record = {
        'experiment': exp_label,
        'system': sys_id,
        'replicate': extract_replicate(sys_id)
    }
    
    # Add portable water aggregates
    record.update(port_metrics)
    # Add nutrient aggregates
    record.update(nut_metrics)
    # Add global sensor aggregates
    record.update(sensor_aggregates.get(exp_label, {}))
    # Add seedling baselines
    record.update(seedling_baselines.get(exp_label, {}))
    
    sys_records.append(record)
    
sys_features_df = pd.DataFrame(sys_records)
print(f"Created system-level features DataFrame: {sys_features_df.shape}")

# 3. Merge system features onto individual harvest plants
# Merge keys: experiment, system, replicate
final_ml_df = pd.merge(
    combined_harvest_df, 
    sys_features_df, 
    on=['experiment', 'system', 'replicate'], 
    how='left'
)
print(f"\nFinal Merged Machine Learning Dataset shape: {final_ml_df.shape}")

Created system-level features DataFrame: (18, 37)

Final Merged Machine Learning Dataset shape: (216, 46)


## 10 · Handle Missing Values Carefully

Let's inspect if there are any missing values in the merged dataset and address them with explicit rationale, rather than blindly filling.

In [10]:
print("--- Missing Values Count per Column in Merged Dataset ---")
null_counts = final_ml_df.isnull().sum()
display(null_counts[null_counts > 0])

--- Missing Values Count per Column in Merged Dataset ---


total_acid_consumption_ml    72
dtype: int64

### 10.1 Missing Values Decisions & Rationale

From the missingness analysis, we observe:
- `total_acid_consumption_ml` has **72 missing values** (representing all plants of **Experiment 2**).
  - *Reasoning*: Experiment 2 raw sheets did not contain an acid consumption sheet. In hydroponics management, acid is only added when water pH goes out of target bounds. Since pH in Experiment 2 was stable or acid dosing was not logged, we will fill these missing values with `0.0` milliliters, signifying that no acid addition was logged during this trial. This preserves the column's numerical nature for ML.
  
Let's execute the missing value filling step.

In [11]:
# Fill missing Experiment 2 acid consumption with 0.0
final_ml_df['total_acid_consumption_ml'] = final_ml_df['total_acid_consumption_ml'].fillna(0.0)

# Verify that no missing values remain
remaining_nulls = final_ml_df.isnull().sum()
print(f"Total missing values remaining in dataset: {remaining_nulls.sum()}")

Total missing values remaining in dataset: 0


## 11 · Save Final Dataset

We save the final flat machine learning dataset to `../data/processed/final_ml_dataset.csv`. This dataset is ready for training models in subsequent phases.

In [12]:
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "final_ml_dataset.csv")

final_ml_df.to_csv(output_path, index=False)
print(f"Successfully saved final ML dataset of shape {final_ml_df.shape} to:")
print(output_path)

Successfully saved final ML dataset of shape (216, 46) to:
../data/processed\final_ml_dataset.csv


## 12 · Create Feature Engineering Report

We programmatically write out a report detailing the dataset statistics, the target variable, the final list of features, and the missing value handling summary to `../reports/feature_engineering_report.md`.

In [13]:
report_dir = "../reports"
os.makedirs(report_dir, exist_ok=True)
report_path = os.path.join(report_dir, "feature_engineering_report.md")

# Compile lists of targets and features
target_cols = ['target_total_weight_g']
metadata_cols = ['experiment', 'system', 'replicate', 'plant_no']
feature_cols = [c for c in final_ml_df.columns if c not in target_cols + metadata_cols]

report_md = f"""# HydroGrow AI — Feature Engineering Report

**Generated Date:** 2026-07-15  
**Phase:** Phase 3 (Feature Engineering)  
**Status:** Completed successfully  

## 1. Dataset Overview
- **Number of Samples (Rows):** {len(final_ml_df)} plants
- **Number of Features (Columns):** {len(feature_cols)} features
- **Target Variable:** `{target_cols[0]}` (Fresh weight in grams at harvest)

## 2. Feature Standardizations and Mappings
Replicate systems from sheet-level datasets (`replicate_X_tY`) were mapped to harvest systems as follows:
- **Experiment 1**: Replicate X, Tank Y maps to `R{{Y}}-T{{X}}`
- **Experiment 2**: Replicate X, Tank Y maps to `R{{X}}-T{{Y}}`
- **Experiment 3**: Replicate X, Tank Y maps to `R{{X}}T{{Y}}`

## 3. List of Engineered Features
### 3.1 Metadata & Key Identifiers
"""
for c in metadata_cols:
    report_md += f"- `{c}`\n"
    
report_md += """
### 3.2 Target Outcome Variables
"""
for c in target_cols + ['harvest_plant_height_cm', 'harvest_shoot_weight_g', 'harvest_root_weight_g', 'harvest_root_length_cm', 'harvest_no_of_leaves', 'head_diameter_average_cm', 'canopy_area_cm2']:
    report_md += f"- `{c}`\n"

report_md += """
### 3.3 Environmental Features (Air Parameters)
"""
env_features = [c for c in feature_cols if c.startswith('env_')]
for c in sorted(env_features):
    report_md += f"- `{c}`\n"

report_md += """
### 3.4 Water Quality Features (Root Zone)
"""
water_features = [c for c in feature_cols if c.startswith('water_')]
for c in sorted(water_features):
    report_md += f"- `{c}`\n"

report_md += """
### 3.5 Management Input Features
"""
mgmt_features = [c for c in feature_cols if c.startswith('total_')]
for c in sorted(mgmt_features):
    report_md += f"- `{c}`\n"

report_md += """
### 3.6 Starting Baseline Features
"""
seedling_features = [c for c in feature_cols if c.startswith('initial_')]
for c in sorted(seedling_features):
    report_md += f"- `{c}`\n"

report_md += """
## 4. Missing Value Treatment Summary
- **Summary Rows Exclusion**: Dropped rows in the harvest sheet with `plant_no = NaN` (these were system-level averages included in raw files). This resulted in exactly **216 clean, complete individual plant rows** (72 per experiment).
- **Acid Consumption (Experiment 2)**: Missing `total_acid_consumption_ml` for Experiment 2 (72 records) was filled with **`0.0`**. Acid additions were not logged or needed during this trial, making 0.0 the correct physical representation.
- **Water Quality (pH, EC, TDS, Water Temp)**: Highly complete; 0% missingness after merging, showing successful temporal aggregation and key mapping.
- **No other missing values exist in the final ML dataset.**
"""

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_md)

print(f"Successfully wrote Feature Engineering Report to:")
print(report_path)


Successfully wrote Feature Engineering Report to:
../reports\feature_engineering_report.md


In [14]:
import pandas as pd

df = pd.read_csv("../data/processed/final_ml_dataset.csv")

print(df.shape)

df.head()

(216, 46)


,experiment,system,plant_no,target_total_weight_g,harvest_plant_height_cm,harvest_shoot_weight_g,harvest_root_weight_g,harvest_root_length_cm,harvest_no_of_leaves,head_diameter_average_cm,...,env_humidity_min,env_humidity_max,env_humidity_std,env_co2_mean,env_co2_min,env_co2_max,env_co2_std,initial_height_mean_cm,initial_weight_mean_g,initial_root_length_mean_cm
0,Experiment 1,R1-T1,1,243.0,42.0,208.0,33.0,28.5,30.0,25.50,...,41.16,67.65,5.681418,456.084,401.85,554.7,31.700287,9.75,1.224,7.1
1,Experiment 1,R1-T1,2,241.0,54.0,189.0,41.0,43.5,34.0,27.00,...,41.16,67.65,5.681418,456.084,401.85,554.7,31.700287,9.75,1.224,7.1
2,Experiment 1,R1-T1,3,251.0,62.0,208.0,39.0,48.5,34.0,26.25,...,41.16,67.65,5.681418,456.084,401.85,554.7,31.700287,9.75,1.224,7.1
3,Experiment 1,R1-T1,4,200.0,55.0,183.0,15.0,41.5,26.0,26.00,...,41.16,67.65,5.681418,456.084,401.85,554.7,31.700287,9.75,1.224,7.1
4,Experiment 1,R1-T1,5,200.0,66.0,174.0,22.0,52.0,25.0,27.50,...,41.16,67.65,5.681418,456.084,401.85,554.7,31.700287,9.75,1.224,7.1
